# Random Forest Classifier

In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
# imports and path setup
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import tqdm
from sklearn.utils import shuffle
from joblib import Parallel, delayed
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from userkits.features import *
from userkits.utils import *

In [14]:
HALF_SIZE = True

In [15]:
# load data from train and eval directories
# set half=True to resize images to half to reduce memory usage
X, y = load_train_data(data_dir='../train_data', half=HALF_SIZE)
X, y = shuffle(X, y, random_state=42)
























































































































































































































































Loading train data: 100%|██████████| 30/30 [00:22<00:00,  1.34it/s]


In [10]:
def extract_features(images):
    features_list = []
    def process_image(img):
        feats = []
        # add feature functions here
        feats.extend(color_histogram(img))
        feats.extend(lbp_texture_features(img))
        feats.extend(find_mean(img))
        feats.extend(find_stddev(img))
        feats.extend(edge_density(img))
        feats.extend(shannon_entropy(img))
        feats.extend(brightness(img))
        feats.extend(green_pixel_ratio(img))
        
        return feats

    features_list = Parallel(n_jobs=-1)(delayed(process_image)(img) for img in tqdm.tqdm(images, desc="Extracting features"))
    return np.array(features_list)

In [11]:
X_features = extract_features(X)
X_features.shape

Extracting features:   1%|          | 10/1483 [00:00<00:58, 25.01it/s]

TypeError: 'numpy.float64' object is not iterable

Extracting features:   1%|          | 10/1483 [00:15<00:58, 25.01it/s]

In [12]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X_features, y_encoded, test_size=0.2) # you can change test_size
clf = RandomForestClassifier() # you can tune hyperparameters here
clf.fit(X_train, y_train)
print("Train Accuracy:", clf.score(X_train, y_train))
print("Test Accuracy:", clf.score(X_test, y_test))

NameError: name 'X_features' is not defined

## Evaluate

In [ ]:
# load eval data
# set half=True to resize images to half to reduce memory usage
X_eval, file_ids = load_eval_data("../eval_data", half=HALF_SIZE) 

In [ ]:
X_eval_features = extract_features(X_eval)
eval_predictions = clf.predict(X_eval_features)
print(eval_predictions[:5])

In [ ]:
try:
    preds = label_encoder.inverse_transform(eval_predictions)
except Exception:
    preds = eval_predictions

save_predictions(preds, file_ids, output_file='../output/rf_predictions.csv')